# Urban Air Quality Analysis, Statistical Visualization and PM2.5 Prediction Using Deep Learning
**Course Code:** U21ADP05 — Exploratory Data Analysis and Visualization  
**Institution:** KPR Institute of Engineering and Technology  
**Department:** Artificial Intelligence and Data Science  
**Author:** [YOUR NAME] | **Roll No:** [YOUR ROLL NUMBER]  
**Dataset:** Beijing Multi-Site Air Quality Data (UCI Machine Learning Repository, 12 Stations, 420,768 records, 18 features)  

---

## 1. Project Overview & Research Objectives
Urban air pollution is a premier global environmental health risk. Fine particulate matter ($PM_{2.5}$) penetrates deep into the human respiratory and circulatory systems, causing cardiovascular mortality, asthma exacerbation, and systemic inflammation.

### Primary Research Question:
> **Can past criteria pollutant measurements and environmental conditions accurately predict $PM_{2.5}$ concentration one hour ahead ($t+1$) across multi-site urban monitoring stations without temporal data leakage?**


In [1]:
import os
import sys
import glob
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import norm, lognorm, gamma, weibull_min, kstest
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import Image, display

# Add src to sys.path
sys.path.append(os.path.abspath('../src'))
from data_preprocessing import load_raw_dataset, clean_and_impute_data, engineer_features, create_chronological_splits
from train_model import AirQualityMLP, get_feature_columns

sns.set_theme(style='whitegrid', font_scale=1.1)
%matplotlib inline
print("Libraries imported successfully!")


No event loop hook running.
Libraries imported successfully!


## 2. Dataset Ingestion & Data Quality Audit
We load all 12 nationally-controlled air quality monitoring stations in Beijing from the official UCI repository archive.


In [2]:
raw_dir = os.path.abspath('../data/raw')
df_raw = load_raw_dataset(raw_dir)
print(f"Total Consolidated Records: {len(df_raw):,}")
print(f"Total Features: {df_raw.shape[1]}")
print(f"Unique Monitoring Stations ({df_raw['station'].nunique()}): {df_raw['station'].unique().tolist()}")
df_raw.head()


Total Consolidated Records: 420,768
Total Features: 18
Unique Monitoring Stations (12): ['Aotizhongxin', 'Changping', 'Dingling', 'Dongsi', 'Guanyuan', 'Gucheng', 'Huairou', 'Nongzhanguan', 'Shunyi', 'Tiantan', 'Wanliu', 'Wanshouxigong']
Out[0]: 
   No  year  month  day  hour  ...  DEWP  RAIN   wd  WSPM       station
0   1  2013      3    1     0  ... -18.8   0.0  NNW   4.4  Aotizhongxin
1   2  2013      3    1     1  ... -18.2   0.0    N   4.7  Aotizhongxin
2   3  2013      3    1     2  ... -18.2   0.0  NNW   5.6  Aotizhongxin
3   4  2013      3    1     3  ... -19.4   0.0   NW   3.1  Aotizhongxin
4   5  2013      3    1     4  ... -19.5   0.0    N   2.0  Aotizhongxin

[5 rows x 18 columns]


   No  year  month  day  hour  ...  DEWP  RAIN   wd  WSPM       station
0   1  2013      3    1     0  ... -18.8   0.0  NNW   4.4  Aotizhongxin
1   2  2013      3    1     1  ... -18.2   0.0    N   4.7  Aotizhongxin
2   3  2013      3    1     2  ... -18.2   0.0  NNW   5.6  Aotizhongxin
3   4  2013      3    1     3  ... -19.4   0.0   NW   3.1  Aotizhongxin
4   5  2013      3    1     4  ... -19.5   0.0    N   2.0  Aotizhongxin

[5 rows x 18 columns]

### Missing Value Inventory & Data Integrity
We audit missing observations per feature across the 420,768 records.


In [3]:
missing_counts = df_raw.isnull().sum()
missing_pct = (missing_counts / len(df_raw)) * 100
quality_df = pd.DataFrame({
    'Feature': df_raw.columns,
    'Data_Type': df_raw.dtypes,
    'Missing_Count': missing_counts,
    'Missing_Pct (%)': missing_pct.round(3)
}).sort_values(by='Missing_Pct (%)', ascending=False)
display(quality_df)
print(f"Total Exact Duplicate Records: {df_raw.duplicated().sum()}")


         Feature Data_Type  Missing_Count  Missing_Pct (%)
CO            CO   float64          20701            4.920
O3            O3   float64          13277            3.155
NO2          NO2   float64          12116            2.879
SO2          SO2   float64           9021            2.144
PM2.5      PM2.5   float64           8739            2.077
PM10        PM10   float64           6449            1.533
wd            wd    object           1822            0.433
DEWP        DEWP   float64            403            0.096
TEMP        TEMP   float64            398            0.095
PRES        PRES   float64            393            0.093
RAIN        RAIN   float64            390            0.093
WSPM        WSPM   float64            318            0.076
year        year     int64              0            0.000
No            No     int64              0            0.000
day          day     int64              0            0.000
month      month     int64              0            0.0

## 3. Data Cleaning & Temporal Imputation
Sensor dropouts and maintenance windows are addressed using station-wise continuous time-series linear interpolation, preserving physical autocorrelation without arbitrary data loss.


In [4]:
df_clean = clean_and_impute_data(df_raw)
print(f"Remaining Missing Values after Temporal Imputation: {df_clean.isnull().sum().sum()}")
df_clean.describe().T


Remaining Missing Values after Temporal Imputation: 0
Out[0]: 
             count  ...           std
No        420768.0  ...  10122.116943
year      420768.0  ...      1.177198
month     420768.0  ...      3.448707
day       420768.0  ...      8.800102
hour      420768.0  ...      6.922195
PM2.5     420768.0  ...     80.964736
PM10      420768.0  ...     92.446817
SO2       420768.0  ...      21.92062
NO2       420768.0  ...     35.374539
CO        420768.0  ...   1160.922359
O3        420768.0  ...     56.787684
TEMP      420768.0  ...     11.437867
PRES      420768.0  ...     10.474302
DEWP      420768.0  ...     13.797675
RAIN      420768.0  ...      0.820638
WSPM      420768.0  ...      1.246674
datetime    420768  ...           NaN

[17 rows x 8 columns]


             count  ...           std
No        420768.0  ...  10122.116943
year      420768.0  ...      1.177198
month     420768.0  ...      3.448707
day       420768.0  ...      8.800102
hour      420768.0  ...      6.922195
PM2.5     420768.0  ...     80.964736
PM10      420768.0  ...     92.446817
SO2       420768.0  ...      21.92062
NO2       420768.0  ...     35.374539
CO        420768.0  ...   1160.922359
O3        420768.0  ...     56.787684
TEMP      420768.0  ...     11.437867
PRES      420768.0  ...     10.474302
DEWP      420768.0  ...     13.797675
RAIN      420768.0  ...      0.820638
WSPM      420768.0  ...      1.246674
datetime    420768  ...           NaN

[17 rows x 8 columns]

## 4. Continuous & Discrete Probability Distribution Fitting
We fit theoretical candidate continuous probability distributions (Normal, Lognormal, Gamma, and Weibull) to hourly $PM_{2.5}$ measurements and evaluate goodness-of-fit via Log-Likelihood, AIC, BIC, and Kolmogorov-Smirnov test statistics. In addition, we evaluate discrete count distributions for monthly severe exceedance days ($PM_{2.5} > 75\ \mu g/m^3$) across 576 station-months to assess overdispersion and compare Poisson vs. Negative Binomial models.


In [5]:
print("Continuous Distribution Goodness-of-Fit (PM2.5):")
dist_fits = pd.read_csv('../outputs/continuous_distribution_fits.csv')
display(dist_fits)

print("\nDiscrete Distribution Fit Comparison (Monthly Exceedances):")
with open('../outputs/discrete_distribution_summary.json', 'r') as f:
    disc_summary = json.load(f)
for k, v in disc_summary.items():
    print(f"  {k}: {v}")


Continuous Distribution Goodness-of-Fit (PM2.5):
  Distribution                Parameters  ...  KS_Statistic    KS_p_value
0       Normal          79.6320, 80.8958  ...        0.1716  0.000000e+00
1    Lognormal  1.0925, -0.9162, 48.3599  ...        0.0571  4.445763e-57
2        Gamma   0.9493, 2.0000, 81.8099  ...        0.0355  2.191611e-22
3      Weibull   0.9588, 2.0000, 76.1675  ...        0.0325  9.493453e-19

[4 rows x 7 columns]

Discrete Distribution Fit Comparison (Monthly Exceedances):
  variable: Monthly Exceedance Days (PM2.5 > 75 ug/m3)
  sample_size_station_months: 576
  sample_mean: 12.267
  sample_variance: 18.961
  dispersion_ratio (Var/Mean): 1.546
  poisson_fit: {'lambda': 12.267, 'log_likelihood': -1709.57, 'aic': 3421.15, 'bic': 3425.51}
  negative_binomial_fit: {'r_dispersion_param': 19.785, 'p_probability': 0.617, 'log_likelihood': -1674.64, 'aic': 3353.27, 'bic': 3361.98}
  comparison_summary: Overdispersion is present (Var/Mean = 1.546 > 1.0). Formal compariso

## 5. Statistical Hypothesis Testing
Formal statistical hypothesis tests are conducted to evaluate:
1. Spatial heterogeneity across 12 monitoring stations (Kruskal-Wallis H-test).
2. Seasonal variation across Spring, Summer, Autumn, and Winter (One-Way ANOVA).
3. Winter vs. Summer contrast with effect size (Mann-Whitney U test and Cohen's d).


In [6]:
with open('../outputs/hypothesis_testing_results.json', 'r') as f:
    hypo_res = json.load(f)

for k, v in hypo_res.items():
    print(f"=== {v['test']} ===")
    for prop, val in v.items():
        if prop != 'test':
            print(f"  {prop}: {val}")
    print()


=== Kruskal-Wallis H-test across 12 Monitoring Stations ===
  statistic: 3548.99
  p_value: 0.0
  conclusion: Significant spatial heterogeneity across monitoring stations (p < 0.001).

=== One-Way ANOVA across 4 Seasons for PM2.5 ===
  F_statistic: 2710.16
  p_value: 0.0
  conclusion: Significant seasonal variation in PM2.5 concentrations (p < 0.001).

=== Mann-Whitney U Test: Winter vs Summer PM2.5 ===
  U_statistic: 117171606.0
  p_value: 4.694922944602535e-10
  winter_mean: 95.78
  summer_mean: 64.97
  winter_median: 55.0
  summer_median: 51.0
  cohens_d: 0.36
  conclusion: Winter PM2.5 (mean=95.8 ug/m3) is significantly higher than Summer (mean=65.0 ug/m3), with medium-to-large effect size (d=0.360, p < 0.001).



## 6. The Seven Stages of Visualization (Fry, 2008)
| Stage | Process | Concrete Implementation in this Project |
|---|---|---|
| **1. Acquire** | Ingestion | Consolidated 12 multi-site CSV files from UCI repository (`420,768` records). |
| **2. Parse** | Structure | Formed ISO-8601 Datetime index from year, month, day, hour; enforced continuous hourly frequency. |
| **3. Filter** | Curation | Isolated criteria pollutants, clipped non-physical negative sensor spikes, handled calibration dropouts. |
| **4. Mine** | Pattern Extraction | Generated correlation matrices, non-parametric percentiles, parametric distribution parameters, lagged cross-correlations. |
| **5. Represent** | Visual Mapping | Selected KDE plots, boxplots, heatmaps, dual-axis diurnal trends, and residual distributions. |
| **6. Refine** | Polish | Formatted publication axes with physical units ($\mu g/m^3$, $hPa$, $^\circ C$, $m/s$), color-blind friendly palettes, and reference thresholds. |
| **7. Interact** | Dynamic Querying | Built interactive Plotly dashboard (`fig8_interactive_air_quality.html`) featuring multi-pollutant hover cards and time sliders. |


### Key Exploratory Visualizations


In [7]:
print("Fig 2: Continuous Probability Distribution Fits for PM2.5")
display(Image(filename='../figures/fig2_pm25_distribution_and_fits.png'))

print("Fig 3: Multi-Station Spatial Distribution")
display(Image(filename='../figures/fig3_pollutant_boxplots_by_station.png'))

print("Fig 4: Correlation Heatmap Across Criteria Pollutants & Weather")
display(Image(filename='../figures/fig4_correlation_heatmap.png'))

print("Fig 5: Seasonal & Diurnal Dynamic Profiles")
display(Image(filename='../figures/fig5_seasonal_and_diurnal_profiles.png'))

print("Fig 6: Meteorological Dispersion & Inversion Dependencies")
display(Image(filename='../figures/fig6_pm25_vs_meteorology.png'))


Fig 2: Continuous Probability Distribution Fits for PM2.5
_Ga=T,f=100,m=1;iVBORw0KGgoAAAANSUhEUgAAC7gAAAcICAYAAACsf4x4AAAAOnRFWHRTb2Z0d2FyZQBNYXRwbG90bGliIHZlcnNpb24zLjExLjIsIGh0dHBzOi8vbWF0cGxvdGxpYi5vcmcvgI3uAAAAAAlwSFlzAAAuIwAALiMBeKU/dgABAABJREFUeJzs3QeYXGX5N+A3IQkBQkIgQOi9Q+i9CgLSIh1FihQBBUQEBKUoSFGqokiTJiC9q/QmRekQemihhRJKgAAh9bue8/923d3szM7uzuycmbnv69ormXbmzJl3TnnP7zxvr6lTp05NAAAAAAAAAAAAAABQZb2rPQMAAAAAAAAAAAAAABAE3AEAAAAAAAAAAAAAyAUBdwAAAAAAAAAAAAAAckHAHQAAAAAAAAAAAACAXBBwBwAAAAAAAAAAAAAgFwTcAQAAAAAAAAAAAADIBQF3AAAAAAAAAAAAAAByQcAdAAAAAAAAAAAAAIBcEHAHAAAAAAAAAAAAACAXBNwBAAAAAAAAAAAAAMgFAXcAAAAAAAAAAAAAAHJBwB0AAAAAAAAAAAAAgFwQcAcAAAAAAAAAAAAAIBcE3AEAAAAAAAAAAAAAyAUBdwAAAAAAAAAAAAAAckHAHQAAAAAAAAAAAACAXBBwBwAAAAAAAAAAAAAgFwTcAQAAAAAAAAAAAADIBQF3AAAAAAAAAAAAAAByQcAdAAAAAAAAAAAAAIBcEHAHAAAAAAAAAAAAACAXBNwBAAAAAAAAAAAAAMgFAXcAAAAAAAAAAAAAAHJBwB0AAAAAAAAAAAAAgFwQcAcAAAAAAAAAAAAAIBcE3AEAAAAAAAAAAAAAyAUBdwAAAAAAAAAAAAAAckHAHQAAAAAAAAAAAACAXBBwBwAAAAAAAAAAAAAgFwTcAQAAAAAAAAAAAADIBQF3A

## 7. Feature Engineering & Strict Chronological Splitting
To prevent future-data contamination, we generate past-only lag features ($t, t-1, t-2, t-5$), multi-pollutant lags, 6-hour and 24-hour moving averages, and cyclical trigonometric hour/month encodings.

Data is partitioned strictly by time:
- **Train Period:** 2013-03-01 to 2016-02-29 (36 months, ~75%)
- **Validation Period:** 2016-03-01 to 2016-10-31 (8 months, ~17%)
- **Held-Out Test Period:** 2016-11-01 to 2017-02-28 (4 months, ~8% - severe winter period)


In [8]:
with open('../outputs/dataset_split_meta.json', 'r') as f:
    split_meta = json.load(f)

print(f"Training Samples:   {split_meta['train_samples']:,} ({split_meta['train_date_range'][0]} to {split_meta['train_date_range'][1]})")
print(f"Validation Samples: {split_meta['val_samples']:,} ({split_meta['val_date_range'][0]} to {split_meta['val_date_range'][1]})")
print(f"Test Samples:       {split_meta['test_samples']:,} ({split_meta['test_date_range'][0]} to {split_meta['test_date_range'][1]})")
print(f"Number of Engineered Input Predictors: {len(split_meta['features'])}")


Training Samples:   315,588 (2013-03-01 05:00:00 to 2016-02-29 23:00:00)
Validation Samples: 70,560 (2016-03-01 00:00:00 to 2016-10-31 23:00:00)
Test Samples:       34,548 (2016-11-01 00:00:00 to 2017-02-28 22:00:00)
Number of Engineered Input Predictors: 23


## 8. Deep Learning Model Training & Hyperparameter Tuning
We train a Multi-Layer Perceptron (MLP) implemented in PyTorch with Batch Normalization, Dropout regularization, AdamW optimizer, and ReduceLROnPlateau learning rate scheduling.


In [9]:
with open('../outputs/hyperparameter_tuning.json', 'r') as f:
    tuning_df = pd.DataFrame(json.load(f))
display(tuning_df)


  config_name   hidden_layers  learning_rate  val_mse  val_rmse  epochs_trained
0   MLP_Small        [64, 32]         0.0010  184.099    13.568              16
1  MLP_Medium   [128, 64, 32]         0.0010  186.843    13.669               9
2   MLP_Large  [256, 128, 64]         0.0005  182.715    13.517              17


## 9. Model Evaluation on Held-Out Test Set
We compare the tuned Deep Learning MLP against two established baselines:
1. **Persistence Baseline:** $\hat{y}_{t+1} = y_t$
2. **Ridge Regression:** Linear regularized baseline


In [10]:
eval_df = pd.read_csv('../outputs/model_evaluation_metrics.csv')
display(eval_df)


                           Model  MAE (ug/m3)  RMSE (ug/m3)  R2_Score  MAPE (%)
0           Persistence Baseline       13.144        26.116    0.9436     32.71
1               Ridge Regression       12.350        24.199    0.9516     35.25
2  Deep Learning MLP (MLP_Large)       11.865        22.797    0.9571     30.91


### Model Diagnostic Figures


In [11]:
print("Fig 9: Training & Validation Loss Curves")
display(Image(filename='../figures/fig9_loss_vs_epoch.png'))

print("Fig 10: Actual vs. Predicted PM2.5 on Held-Out Test Set")
display(Image(filename='../figures/fig10_actual_vs_predicted.png'))

print("Fig 11: Residual Error Distribution")
display(Image(filename='../figures/fig11_residual_distribution.png'))

print("Fig 13: Model Comparison Metrics")
display(Image(filename='../figures/fig13_model_comparison_metrics.png'))

print("Fig 14: Permutation Feature Importance")
display(Image(filename='../figures/fig14_permutation_feature_importance.png'))


Fig 9: Training & Validation Loss Curves
_Ga=T,f=100,m=1;iVBORw0KGgoAAAANSUhEUgAACfYAAAXcCAYAAABqUE26AAAAOnRFWHRTb2Z0d2FyZQBNYXRwbG90bGliIHZlcnNpb24zLjExLjIsIGh0dHBzOi8vbWF0cGxvdGxpYi5vcmcvgI3uAAAAAAlwSFlzAAAuIwAALiMBeKU/dgABAABJREFUeJzs3QXYFOX+//Gb7pYGFVBpBQNRLECxAVuOia1YGMdGxG49FtiIjYXdgqBHsUABAekOQbrjf33u33+fszEzO9uzu+/Xde0Fzz67O/PM7MzO3vOZ77fU9u3btxsAAAAAAAAAAAAAAAAAABAIpXM9AwAAAAAAAAAAAAAAAAAA4H8I9gEAAAAAAAAAAAAAAAAAECAE+wAAAAAAAAAAAAAAAAAACBCCfQAAAAAAAAAAAAAAAAAABAjBPgAAAAAAAAAAAAAAAAAAAoRgHwAAAAAAAAAAAAAAAAAAAUKwDwAAAAAAAAAAAAAAAACAACHYBwAAAAAAAAAAAAAAAABAgBDsAwAAAAAAAAAAAAAAAAAgQAj2AQAAAAAAAAAAAAAAAAAQIAT7AAAAAAAAAAAAAAAAAAAIEIJ9AAAAAAAAAAAAAAAAAAAECME+AAAAAAAAAAAAAAAAAAAChGAfAAAAAAAAAAAAAAAAAAABQrAPAAAAAAAAAAAAAAAAAIAAIdgHAAAAAAAAAAAAAAAAAECAEOwDAAAAAAAAAAAAAAAAACBACPYBAAAAAAAAAAAAAAAAABAgBPsAAAAAAAAAAAAAAAAAAAgQgn0AAAAAAAAAAAAAAAAAAAQIwT4AAAAAAAAAAAAAAAAAAAKEYB8AAAAAAAAAAAAAAAAAAAFCsA8AAAAAAAAAAAAAAAAAgAAh2AcAAAAAAAAAAAAAAAAAQIAQ7AMAAAAAAAAAAAAAAAAAIEAI9gEAAAAAAAAAAAAAAAAAEC

## 10. Conclusions & Future Scope
### Key Findings:
1. **Autoregressive and Co-pollutant Dominance:** Past fine particles ($PM_{2.5\_lag1}$, $PM_{2.5\_lag2}$) and coarse particles ($PM_{10\_lag1}$) are the strongest predictors of next-hour concentrations.
2. **Distribution Characteristics:** Continuous PM2.5 is best fitted by a Weibull distribution (AIC = 214,023.2, KS = 0.0325) closely followed by Gamma, while monthly exceedance counts exhibit overdispersion (Var/Mean = 1.55) supporting a Negative Binomial model over Poisson ($\Delta$AIC = -67.9).
3. **Deep Learning Superiority:** The PyTorch MLP achieved a test RMSE of **22.80 $\mu g/m^3$**, an MAE of **11.87 $\mu g/m^3$**, an $R^2$ of **0.9571**, and a mean residual error of **1.71 $\mu g/m^3$** (std = 22.73 $\mu g/m^3$, median = 0.58 $\mu g/m^3$), outperforming Ridge regression ($RMSE = 24.20 \mu g/m^3$) and Persistence baseline ($RMSE = 26.12 \mu g/m^3$).

### Future Work:
- Integrating recurrent or spatio-temporal architectures (LSTM/GRU/GNN) to explicitly model spatial graph topologies across monitoring stations.
- Incorporating boundary-layer height and satellite aerosol optical depth (AOD) retrievals.
